# ComfyUI on Colab, driven from your local machine

Run cells 1 to 5 in order. Cell 4 prints the address you open in your own
browser. Leave cell 4 running for the whole session.

**Before the first run**

1. Runtime -> Change runtime type. Start on **T4** or **L4** for image work.
   Switch to **A100 High-RAM** only when you move to LTX-2.5 video.
2. In the secrets panel on the left, add `TS_AUTHKEY` (Tailscale auth key) and
   `HF_TOKEN` (Hugging Face token), both with notebook access enabled.
3. Make this repository reachable: paste a GitHub URL below, or copy the
   project folder to `MyDrive/colab-comfy/repo`.

In [ ]:
#@title 1. Configure, mount Drive, load secrets { display-mode: "form" }
GIT_URL    = ""  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/colab-comfy"  #@param {type:"string"}
TUNNEL     = "tailscale"  #@param ["tailscale", "cloudflared", "none"]
TS_HOSTNAME = "colab-comfy"  #@param {type:"string"}
COMFY_EXTRA_ARGS = ""  #@param {type:"string"}

import os, pathlib, subprocess, textwrap

from google.colab import drive
drive.mount('/content/drive')

REPO = "/content/colab-comfy"
if GIT_URL:
    if os.path.isdir(REPO + "/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", GIT_URL, REPO], check=True)
elif os.path.isdir(f"{DRIVE_ROOT}/repo/scripts"):
    subprocess.run(["rsync", "-a", f"{DRIVE_ROOT}/repo/", REPO + "/"], check=True)
else:
    raise SystemExit(
        f"No repo found. Set GIT_URL, or copy the project folder to {DRIVE_ROOT}/repo"
    )

from google.colab import userdata
for key in ("TS_AUTHKEY", "HF_TOKEN"):
    try:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f"loaded secret {key}")
    except Exception:
        print(f"secret {key} not set")

pathlib.Path(REPO, "config", "comfy.env").write_text(textwrap.dedent(f'''
    COMFY_ROOT=/content/ComfyUI
    DRIVE_ROOT={DRIVE_ROOT}
    MODEL_CACHE={DRIVE_ROOT}/models_cache
    COMFY_PORT=8188
    TUNNEL={TUNNEL}
    TS_HOSTNAME={TS_HOSTNAME}
    SYNC_INTERVAL=30
    COMFY_EXTRA_ARGS={COMFY_EXTRA_ARGS}
''').lstrip())

os.environ["REPO"] = REPO
print("repo ready at", REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
#@title 2. Install ComfyUI and custom nodes (~3 min)
!bash $REPO/scripts/bootstrap.sh

In [ ]:
#@title 3. Fetch a model profile { display-mode: "form" }
PROFILE = "smoke"  #@param ["smoke", "flux1-dev", "qwen-image", "ltx-2.5"]
#  smoke       ~2 GB   sanity check, runs on a T4
#  flux1-dev   ~17 GB  images, L4 or better
#  qwen-image  ~30 GB  images with legible text, wants 40 GB VRAM
#  ltx-2.5     ~39 GB  video plus audio, A100 40GB and a gated-repo HF_TOKEN
# Files come from the Drive cache when present, otherwise from Hugging Face,
# and are copied back to the cache. Both routes are timed and compared.
!python3 $REPO/scripts/fetch_models.py $PROFILE

In [ ]:
#@title 4. Launch ComfyUI and open the tunnel (leave running)
!bash $REPO/scripts/launch.sh

In [ ]:
#@title 5. Headless smoke test (run while cell 4 is running)
# Queues one image through the API and reports the output filename.
!python3 $REPO/scripts/smoke_test.py

### Optional: attach your local editor

Run the next cell to expose the runtime to VS Code or Cursor. It prints a
GitHub device code once, then in your editor run
`Remote-Tunnels: Connect to Tunnel` and pick the machine name.

With Tailscale you also get SSH for free: `ssh root@colab-comfy` from any
device on your tailnet, which is the route a CLI agent should take.

In [ ]:
#@title 6. VS Code / Cursor tunnel
!bash $REPO/scripts/remote_dev.sh vscode

In [ ]:
#@title 7. Stop everything and flush outputs to Drive
!bash $REPO/scripts/stop.sh